In [1]:
import sys 
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(
    '../data/raw/macro_data.csv',
    index_col='date',
    parse_dates=True
)

df.head()


,gdp_real,unemployment,cpi,fed_funds_rate
date,,,,
1970-01-01,5300.652,3.9,37.9,8.98
1970-02-01,NaN,4.2,38.1,8.98
1970-03-01,NaN,4.4,38.3,7.76
1970-04-01,5308.164,4.6,38.5,8.10
1970-05-01,NaN,4.8,38.6,7.95


In [2]:
print(df.shape)
print(df.describe())

(674, 4)
           gdp_real  unemployment         cpi  fed_funds_rate
count    224.000000    673.000000  673.000000      674.000000
mean   13133.225250      6.054086  163.459961        4.895326
std     5446.714892      1.714309   77.683222        3.839410
min     5299.672000      3.400000   37.900000        0.050000
25%     7996.333750      4.800000  102.100000        1.627500
50%    12640.618500      5.700000  162.000000        4.940000
75%    17258.689750      7.200000  227.842000        6.787500
max    24111.830000     14.800000  327.460000       19.100000


In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

series = {
    "Real GDP (Billions USD)": df["gdp_real"],
    "Unemployment Rate (%)": df["unemployment"],
    "CPI (Index)": df["cpi"],
    "Federal Funds Rate (%)": df["fed_funds_rate"],
}

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=False,
    subplot_titles=list(series.keys()),
    vertical_spacing=0.08
)

for i, (title, data) in enumerate(series.items(), start=1):
    fig.add_trace(
        go.Scatter(
            x=data.dropna().index,
            y=data.dropna().values,
            mode="lines",
            line=dict(color="#e05c5c", width=1.5),
            name=title,
        ),
        row=i, col=1
    )

fig.update_layout(
    height=900,
    title_text="US Macroeconomic Indicators (1970–2026)",
    title_font_size=16,
    paper_bgcolor="#111111",
    plot_bgcolor="#111111",
    font=dict(color="white"),
    showlegend=False,
)

fig.update_xaxes(gridcolor="#333333", showgrid=True)
fig.update_yaxes(gridcolor="#333333", showgrid=True)

fig.show()
fig.write_html("../outputs/figures/01_macro_overview.html")


In [6]:
from dotenv import load_dotenv
import os
from fredapi import Fred

load_dotenv(dotenv_path=r"C:\Users\Usuario\Documents\Python\US-macro-analysis\.env")
fred = Fred(api_key=os.environ.get("FRED_API_KEY"))

recessions = fred.get_series("USREC", observation_start="1970-01-01")
recessions.index.name = "date"
print(recessions.value_counts())
recessions.head()

0.0    589
1.0     85
Name: count, dtype: int64


date
1970-01-01    1.0
1970-02-01    1.0
1970-03-01    1.0
1970-04-01    1.0
1970-05-01    1.0
dtype: float64

In [7]:
# identifica inicio e fim de cada recessão
rec = recessions.copy()
starts = rec[(rec == 1) & (rec.shift(1) == 0)].index.tolist()
ends = rec[(rec == 0) & (rec.shift(1) == 1)].index.tolist()

# se começou em recessão em 1970, adiciona o início manualmente
if rec.iloc[0] == 1:
    starts = [rec.index[0]] + starts

# garante mesmo número de pares
pairs = list(zip(starts, ends[:len(starts)]))

# gráfico
series = {
    "Real GDP (Billions USD)": df["gdp_real"],
    "Unemployment Rate (%)": df["unemployment"],
    "CPI (Index)": df["cpi"],
    "Federal Funds Rate (%)": df["fed_funds_rate"],
}

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=False,
    subplot_titles=list(series.keys()),
    vertical_spacing=0.08
)

for i, (title, data) in enumerate(series.items(), start=1):
    fig.add_trace(
        go.Scatter(
            x=data.dropna().index,
            y=data.dropna().values,
            mode="lines",
            line=dict(color="#e05c5c", width=1.5),
            name=title,
        ),
        row=i, col=1
    )

# adiciona faixas de recessão em todos os subplots
for start, end in pairs:
    for row in range(1, 5):
        fig.add_vrect(
            x0=start, x1=end,
            fillcolor="gray",
            opacity=0.2,
            layer="below",
            line_width=0,
            row=row, col=1
        )

fig.update_layout(
    height=900,
    title_text="US Macroeconomic Indicators (1970–2026) — Shaded areas: NBER Recessions",
    title_font_size=14,
    paper_bgcolor="#111111",
    plot_bgcolor="#111111",
    font=dict(color="white"),
    showlegend=False,
)

fig.update_xaxes(gridcolor="#333333", showgrid=True)
fig.update_yaxes(gridcolor="#333333", showgrid=True)

fig.show()
fig.write_html("../outputs/figures/01_macro_overview.html")

In [10]:
import os
os.makedirs("../data/processed", exist_ok=True)

df["recession"] = recessions
df.to_csv("../data/processed/macro_data_processed.csv")
print("Saved.")


Saved.
